In [1]:
import h5py
import numpy as np
path = '/work/users/t/i/tis/data/jhu_spatiotemporal/data260518/121405.mat'

with h5py.File(path, 'r') as f:
    print("Top-level entries:", list(f.keys()))

Top-level entries: ['#refs#', '#subsystem#', 'ImgData', 'P', 'RcvData', 'Receive', 'TX', 'Trans', 'activeSparseChannelsByMode', 'frameCycle', 'frameLocalFrame', 'frameSparseMode', 'frameType', 'sparseModeApods']


In [4]:
def load_struct_to_dict(file_path, group_name):
    struct_dict = {}
    with h5py.File(file_path, 'r') as f:
        group = f[group_name]
        
        for key in group.keys():
            data = group[key][:]
            
            # Check if this dataset is an array of references
            if data.dtype.kind == 'O' and data.size > 0 and isinstance(data.flat[0], h5py.Reference):
                # Resolve all references and stack them into one array
                struct_dict[key] = np.stack([np.squeeze(f[ref][:]) for ref in data.flatten()])
            else:
                # Standard array, just squeeze and store
                struct_dict[key] = np.squeeze(data)
                
    return struct_dict

# Usage:
tx_struct = load_struct_to_dict(path, 'TX')

# Now you can easily access any variable as one big numpy array:
print(tx_struct['Apod'].shape)       # (6400, 128)
print(tx_struct['Delay'].shape)      # (6400, 128)
print(tx_struct['Numpulses'].shape)  # (6400,)

(6400, 128)
(6400, 128)
(6400, 256)


In [12]:
print(tx_struct['Apod'][50])
print(tx_struct['Delay'][50])
print(tx_struct['Origin'][64])

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0.]
[0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.15553374
 0.3012499  0.43710592 0.56306191 0.67908069 0.78512784 0.88117177
 0.96718375 1.04313796 1.10901152 1.16478454 1.21044012 1.24596443
 1.27134666 1.28657909 1.29165708 1.28657909 1.27134666 1.24596443
 1.21044012 1.16478454 1.10901152 1.04313796 0

### Compare different Grid Calculations

In [25]:
import numpy as np
from scipy.io import loadmat

# ==========================================
# 1. PREREQUISITES (From your earlier cells)
# ==========================================
# Assuming trans_struct and p_struct are already loaded in your notebook:
# trans_struct = load_struct_to_dict(path, 'Trans')
# p_struct = load_struct_to_dict(path, 'P')

c = 1.54 # assumed speed of sound in mm/us (1540 m/s)
frequency = trans_struct['frequency'] # in MHz
lam = c / frequency # wavelength in mm

# ==========================================
# 2. GRID 1: GROUND TRUTH (From setup.mat PData)
# ==========================================
setup_path = "/proj/yzlinlab/projects/jhu_spatiotemporal/data260421/setup.mat"
setup = loadmat(setup_path, squeeze_me=True, struct_as_record=False)
pdata = setup["PData"]

origin = np.asarray(pdata.Origin)
delta = np.asarray(pdata.PDelta)
size = np.asarray(pdata.Size)
nz, nx, ny = int(size[0]), int(size[1]), int(size[2])

# PData coordinates converted to mm
x_pdata = (origin[0] + np.arange(nx) * delta[0]) * lam
z_pdata = (origin[2] + np.arange(nz) * delta[2]) * lam
X_pdata_mm, Z_pdata_mm = np.meshgrid(x_pdata, z_pdata, indexing="xy")

# ==========================================
# 3. GRID 2: SETUP INFERRED (From setup.mat P)
# ==========================================
p_setup = setup["P"]
setup_start_z = p_setup.startDepth * lam
# Verasonics natively defaults to 0.5 wavelength spacing for Z
setup_z_step = 0.5 * lam 
# Use arange to take exact steps, ignoring endDepth
z_setup_inferred = setup_start_z + np.arange(nz) * setup_z_step

# We use the ground truth x_pdata here to focus purely on the Z-coordinate comparison
X_setup_inferred_mm, Z_setup_inferred_mm = np.meshgrid(x_pdata, z_setup_inferred, indexing="xy")

# ==========================================
# 4. GRID 3: RAW DATA INFERRED (From your HDF5 P and Trans)
# ==========================================
x_inferred = trans_struct['ElementPos'][0, :] * lam 
start_z = p_struct['startDepth'] * lam
# Verasonics natively defaults to 0.5 wavelength spacing for Z
z_step = 0.5 * lam 
# Use arange to take exact steps, ignoring endDepth
z_inferred = start_z + np.arange(nz) * z_step

X_raw_inferred_mm, Z_raw_inferred_mm = np.meshgrid(x_inferred, z_inferred)

# ==========================================
# 5. COMPARISON
# ==========================================
print("--- Grid Shapes ---")
print(f"1. PData Ground Truth: {X_pdata_mm.shape}")
print(f"2. Setup Inferred:     {X_setup_inferred_mm.shape}")
print(f"3. Raw Inferred:       {X_raw_inferred_mm.shape}\n")

def compare_grids(name, X1, Z1, X2, Z2):
    print(f"--- Comparing: PData vs {name} ---")
    if X1.shape != X2.shape:
        print("Cannot perform element-wise comparison: Dimensions do not match.\n")
        return
        
    x_match = np.allclose(X1, X2, atol=1e-5)
    z_match = np.allclose(Z1, Z2, atol=1e-5)
    
    print(f"X-Coordinates Match: {x_match}")
    if not x_match:
        max_x_diff = np.max(np.abs(X1 - X2))
        print(f"  -> Max X Difference: {max_x_diff:.6f} mm")
        
    print(f"Z-Coordinates Match: {z_match}")
    if not z_match:
        max_z_diff = np.max(np.abs(Z1 - Z2))
        print(f"  -> Max Z Difference: {max_z_diff:.6f} mm")
    print()

# Run the comparisons
compare_grids("Setup Inferred (Grid 2)", X_pdata_mm, Z_pdata_mm, X_setup_inferred_mm, Z_setup_inferred_mm)
compare_grids("Raw Data Inferred (Grid 3)", X_pdata_mm, Z_pdata_mm, X_raw_inferred_mm, Z_raw_inferred_mm)

# Sanity check: verify if the parameters in setup.mat match your HDF5 parameters exactly
print("--- Comparing: Setup Inferred vs Raw Inferred ---")
z_setup_raw_match = np.allclose(Z_setup_inferred_mm, Z_raw_inferred_mm, atol=1e-5)
print(f"Z-Coordinates Match: {z_setup_raw_match}")
if not z_setup_raw_match:
    diff = np.max(np.abs(Z_setup_inferred_mm - Z_raw_inferred_mm))
    print(f"  -> Max Difference: {diff:.6f} mm")

--- Grid Shapes ---
1. PData Ground Truth: (374, 128)
2. Setup Inferred:     (374, 128)
3. Raw Inferred:       (374, 128)

--- Comparing: PData vs Setup Inferred (Grid 2) ---
X-Coordinates Match: True
Z-Coordinates Match: True

--- Comparing: PData vs Raw Data Inferred (Grid 3) ---
X-Coordinates Match: True
Z-Coordinates Match: True

--- Comparing: Setup Inferred vs Raw Inferred ---
Z-Coordinates Match: True


In [3]:
def print_shapes(name, obj):
    # Skip MATLAB's internal reference folders
    if not name.startswith('#'):
        # Check if the object is actual data (Dataset)
        if isinstance(obj, h5py.Dataset):
            print(f"{name} shape: {obj.shape}")
        # If it's a folder (Group), just print the name
        elif isinstance(obj, h5py.Group):
            print(f"{name} (Group)")

# Open the file and run the new function
with h5py.File(file_path, 'r') as f:
    print("Data shapes:")
    print("-" * 20)
    f.visititems(print_shapes)

Data shapes:
--------------------
ImgData shape: (1, 1)
P (Group)
P/decFactor shape: (1, 1)
P/endDepth shape: (1, 1)
P/numCycles shape: (1, 1)
P/numDenseFramesPerCycle shape: (1, 1)
P/numFramesPerCycle shape: (1, 1)
P/numSparseModes shape: (1, 1)
P/numTotalFrames shape: (1, 1)
P/numTx shape: (1, 1)
P/startDepth shape: (1, 1)
P/txFocus shape: (1, 1)
RcvData shape: (1, 1)
Receive (Group)
Receive/ADCRate shape: (6400, 1)
Receive/Apod shape: (6400, 1)
Receive/InputFilter shape: (6400, 1)
Receive/LowPassCoef shape: (6400, 1)
Receive/TGC shape: (6400, 1)
Receive/acqNum shape: (6400, 1)
Receive/bufnum shape: (6400, 1)
Receive/callMediaFunc shape: (6400, 1)
Receive/decimFactor shape: (6400, 1)
Receive/decimSampleRate shape: (6400, 1)
Receive/demodFrequency shape: (6400, 1)
Receive/endDepth shape: (6400, 1)
Receive/endSample shape: (6400, 1)
Receive/framenum shape: (6400, 1)
Receive/mode shape: (6400, 1)
Receive/quadDecim shape: (6400, 1)
Receive/sampleMode shape: (6400, 1)
Receive/samplesPerWa

In [20]:
print(trans_struct['elevationFocusMm'])

25.0


In [13]:
import numpy as np

# 1. Constants
c = 1.54 # assumed speed of sound in mm/us (or 1540 m/s)
frequency = tx_struct['Trans']['frequency'] # in MHz
wavelength = c / frequency # in mm

# 2. X-Coordinates (Lateral)
# Grab the first row of ElementPos and convert wavelengths to mm
x_coords = tx_struct['Trans']['ElementPos'][0, :] * wavelength 

# 3. Z-Coordinates (Axial)
start_z = tx_struct['P']['startDepth'] * wavelength
end_z = tx_struct['P']['endDepth'] * wavelength
num_z_pixels = 374 # From your ImgData shape

z_coords = np.linspace(start_z, end_z, num_z_pixels)

# 4. Create the 2D grid
X_grid, Z_grid = np.meshgrid(x_coords, z_coords)

KeyError: 'Trans'

In [4]:
import h5py

file_path = '/work/users/t/i/tis/data/jhu_spatiotemporal/data260518/121405.mat'

with h5py.File(file_path, 'r') as f:
    ref = f['RcvData'][0, 0]
    real_data = f[ref]
    print("Shape:", real_data.shape)
    print("Chunk size:", real_data.chunks)

    ref = f['ImgData'][0, 0]
    img_data = f[ref]
    print("Shape:", img_data.shape)
    print("Chunk size:", img_data.chunks)

Shape: (50, 128, 524288)
Chunk size: (50, 128, 5)
Shape: (50, 1, 128, 374)
Chunk size: (50, 1, 128, 1)


In [ ]:
import h5py

def print_group_values(file_path, group_name):
    """
    Opens an HDF5 file, prints the datasets inside a specific group, 
    and automatically resolves MATLAB's HDF5 object references.
    """
    with h5py.File(file_path, 'r') as f:
        
        if group_name not in f:
            print(f"Error: '{group_name}' not found in the file.")
            return
            
        target_group = f[group_name]
        
        if not isinstance(target_group, h5py.Group):
            print(f"Error: '{group_name}' is a Dataset, not a Group.")
            return

        print(f"Values inside the '{group_name}' Group:")
        print("-" * 30)
        
        for key, item in target_group.items():
            
            if isinstance(item, h5py.Dataset):
                value = item[:]
                
                # 1. Check if this dataset is an array of HDF5 References
                if value.dtype.kind == 'O' and value.size > 0 and isinstance(value.flatten()[0], h5py.Reference):
                    print(f"\n{key}: Array of {value.shape} References")
                    
                    flat_refs = value.flatten()
                    
                    # 2. Limit the output to the first 3 items to prevent notebook crashing
                    num_to_show = min(3, len(flat_refs)) 
                    
                    for i in range(num_to_show):
                        ref = flat_refs[i]
                        try:
                            # 3. Follow the pointer to the actual data
                            real_data = f[ref] 
                            
                            if isinstance(real_data, h5py.Dataset):
                                real_val = real_data[:]
                                # Clean up (1,1) shapes for readability
                                if real_val.shape == (1, 1):
                                    print(f"    ↳ [{i}] resolved: {real_val[0, 0]}")
                                else:
                                    print(f"    ↳ [{i}] resolved: Dataset with shape {real_val.shape}")
                            elif isinstance(real_data, h5py.Group):
                                print(f"    ↳ [{i}] resolved: Nested Group")
                        except Exception:
                            print(f"    ↳ [{i}] resolved: <Invalid Reference>")
                            
                    # Note how many were hidden
                    if len(flat_refs) > num_to_show:
                        print(f"    ↳ ... ({len(flat_refs) - num_to_show} more references omitted)")
                
                # 4. Handle standard datasets (No references)
                else:
                    if value.shape == (1, 1):
                        print(f"{key}: {value[0, 0]}")
                    else:
                        print(f"{key}: \n{value}")
                        
            elif isinstance(item, h5py.Group):
                print(f"{key} is a nested Group (folder).")
                

In [5]:
import h5py
import numpy as np

# Restore NumPy's default print options (restores the '...' truncation)
np.set_printoptions(threshold=1000, edgeitems=3, linewidth=100)

def print_group_values(file_path, group_name):
    """
    Opens an HDF5 file and prints datasets and resolved references 
    horizontally and compactly, just like standard NumPy arrays.
    """
    with h5py.File(file_path, 'r') as f:
        
        if group_name not in f:
            print(f"Error: '{group_name}' not found in the file.")
            return
            
        target_group = f[group_name]
        
        if not isinstance(target_group, h5py.Group):
            print(f"Error: '{group_name}' is a Dataset, not a Group.")
            return

        print(f"Values inside the '{group_name}' Group:")
        print("-" * 30)
        
        for key, item in target_group.items():
            
            if isinstance(item, h5py.Dataset):
                value = item[:]
                
                # Check if this dataset is an array of HDF5 References
                if value.dtype.kind == 'O' and value.size > 0 and isinstance(value.flatten()[0], h5py.Reference):
                    print(f"\n{key}: Array of {value.shape} References")
                    
                    flat_refs = value.flatten()
                    
                    # Limit how many references to show (change to len(flat_refs) to see all 6400)
                    num_to_show = min(5, len(flat_refs)) 
                    
                    for i in range(num_to_show):
                        ref = flat_refs[i]
                        try:
                            real_data = f[ref] 
                            
                            if isinstance(real_data, h5py.Dataset):
                                real_val = real_data[:]
                                
                                # SQUEEZE the array: changes vertical (128, 1) to horizontal (128,)
                                compact_val = np.squeeze(real_val)
                                
                                if compact_val.shape == (): # Scalar value
                                    print(f"    ↳ [{i}] resolved: {compact_val}")
                                else:
                                    # Fix: Create string outside of f-string
                                    array_str = np.array2string(compact_val, separator=', ').replace('\n', '')
                                    print(f"    ↳ [{i}] resolved: {array_str}")
                                    
                            elif isinstance(real_data, h5py.Group):
                                print(f"    ↳ [{i}] resolved: Nested Group")
                        except Exception:
                            print(f"    ↳ [{i}] resolved: <Invalid Reference>")
                            
                    if len(flat_refs) > num_to_show:
                        print(f"    ↳ ... ({len(flat_refs) - num_to_show} more references omitted)")
                            
                # Handle standard datasets (No references)
                else:
                    compact_val = np.squeeze(value)
                    if compact_val.shape == ():
                        print(f"{key}: {compact_val}")
                    else:
                        # Fix: Create string outside of f-string to avoid backslash syntax error
                        array_str = np.array2string(compact_val, separator=', ').replace('\n', '')
                        print(f"{key}: {array_str}")
                        
            elif isinstance(item, h5py.Group):
                print(f"{key} is a nested Group (folder).")

In [7]:
print_group_values(file_path, 'P')

Values inside the 'P' Group:
------------------------------
decFactor: 4.0
endDepth: 192.0
numCycles: 10.0
numDenseFramesPerCycle: 1.0
numFramesPerCycle: 5.0
numSparseModes: 4.0
numTotalFrames: 50.0
numTx: 32.0
startDepth: 5.0
txFocus: 100.0


In [8]:
print_group_values(file_path, 'Receive')

Values inside the 'Receive' Group:
------------------------------

ADCRate: Array of (6400, 1) References
    ↳ [0] resolved: 62.5
    ↳ [1] resolved: 62.5
    ↳ [2] resolved: 62.5
    ↳ [3] resolved: 62.5
    ↳ [4] resolved: 62.5
    ↳ ... (6395 more references omitted)

Apod: Array of (6400, 1) References
    ↳ [0] resolved: [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]
    ↳ [1] resolved: [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1

In [6]:
import h5py
import numpy as np

file_path = '/work/users/t/i/tis/data/jhu_spatiotemporal/data260518/121405.mat'

# The list of top-level datasets you want to inspect
target_vars = [
    'activeSparseChannelsByMode', 
    'frameCycle', 
    'frameLocalFrame', 
    'frameSparseMode', 
    'frameType', 
    'sparseModeApods'
]

with h5py.File(file_path, 'r') as f:
    for var in target_vars:
        # Extract the data and squeeze it (turns shapes like (1, 50) into (50,))
        data = np.squeeze(f[var][:])
        
        print(f"\n--- {var} ---")
        print(f"Shape: {data.shape}")
        
        # Print compactly like a standard NumPy array
        print(np.array2string(data, separator=', '))


--- activeSparseChannelsByMode ---
Shape: (4,)
[<HDF5 object reference>, <HDF5 object reference>, <HDF5 object reference>, <HDF5 object reference>]

--- frameCycle ---
Shape: (50,)
[ 1.,  1.,  1.,  1.,  1.,  2.,  2.,  2.,  2.,  2.,  3.,  3.,  3.,  3.,  3.,  4.,  4.,  4.,  4.,  4.,
  5.,  5.,  5.,  5.,  5.,  6.,  6.,  6.,  6.,  6.,  7.,  7.,  7.,  7.,  7.,  8.,  8.,  8.,  8.,  8.,
  9.,  9.,  9.,  9.,  9., 10., 10., 10., 10., 10.]

--- frameLocalFrame ---
Shape: (50,)
[1., 2., 3., 4., 5., 1., 2., 3., 4., 5., 1., 2., 3., 4., 5., 1., 2., 3., 4., 5., 1., 2., 3., 4., 5.,
 1., 2., 3., 4., 5., 1., 2., 3., 4., 5., 1., 2., 3., 4., 5., 1., 2., 3., 4., 5., 1., 2., 3., 4., 5.]

--- frameSparseMode ---
Shape: (50,)
[0., 1., 2., 3., 4., 0., 1., 2., 3., 4., 0., 1., 2., 3., 4., 0., 1., 2., 3., 4., 0., 1., 2., 3., 4.,
 0., 1., 2., 3., 4., 0., 1., 2., 3., 4., 0., 1., 2., 3., 4., 0., 1., 2., 3., 4., 0., 1., 2., 3., 4.]

--- frameType ---
Shape: (6,)
[3707764736,          2,          1,          1,      

In [9]:
import h5py
import numpy as np

file_path = '/work/users/t/i/tis/data/jhu_spatiotemporal/data260518/121405.mat'

with h5py.File(file_path, 'r') as f:
    
    # 1. Access the Apod dataset inside the TX group
    # Since it has a shape of (6400, 1), we use [1, 0] to get the pointer at index 1
    ref = f['TX/Apod'][120, 0] 
    
    # 2. Follow the pointer to get the actual array data
    actual_array = f[ref][:]
    
    # 3. Squeeze it to flatten (128, 1) into a clean (128,) horizontal array
    clean_array = np.squeeze(actual_array)
    
    print("--- TX/Apod at index 1 ---")
    print(f"Shape: {clean_array.shape}")
    
    # 4. Print it compactly
    np.set_printoptions(threshold=1000, linewidth=120)
    print(np.array2string(clean_array, separator=', ').replace('\n', ''))
    

--- TX/Apod at index 1 ---
Shape: (128,)
[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]
